# **WEB SCRAPING LAMUDI**

## **SCRAPING PERTAMA (MENGUMPULKAN LINK HINGGA HALAMAN KE 70)**

In [15]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [17]:
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://www.google.com/"
}

url = "https://www.lamudi.co.id/jual/"

response = requests.get(url, headers=headers)

print(response.status_code)

202


In [36]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

In [38]:
driver = webdriver.Chrome()

driver.get("https://www.lamudi.co.id/jual/apartemen/")

print(driver.title)

In [39]:
from bs4 import BeautifulSoup

html = driver.page_source

soup = BeautifulSoup(html, "html.parser")

print(soup.title)

<title>Apartemen Dijual di Indonesia | Lamudi</title>


In [28]:
cards = soup.find_all("div", class_="snippet")

print("Jumlah Card:", len(cards))

Jumlah Card: 30


In [29]:
card = cards[0]

print(card.find("span", class_="snippet__content__title"))
print(card.find("div", class_="snippet__content__price"))
print(card.find("div", class_="snippet__content__location"))
print(card.find("a")["href"])

<span class="snippet__content__title" content="Apartemen Dijual di Pancoran" itemprop="name">Apartemen Dijual di Pancoran</span>
<div class="snippet__content__price">
                    Rp 600Jt
            </div>
<div class="snippet__content__location">
<span data-test="snippet-content-location">RW 07, Rawajati, Pancoran, Jakarta Selatan, Daerah Khusus Ibukota Jakarta</span>
</div>
/properti/41032-73-54ce27f8200-7466-19e91bd-aaf4-73c3


In [41]:
import time
import random

In [92]:
START_PAGE = 61
END_PAGE = 70

In [93]:
semua_data = []

driver = webdriver.Chrome()

for page in range(START_PAGE, END_PAGE + 1):

    print(f"Halaman {page}")

    url = f"https://www.lamudi.co.id/jual/apartemen/?page={page}"

    driver.get(url)

    time.sleep(3)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    cards = soup.find_all("div", class_="snippet")

    print(f"Card: {len(cards)}")

    for card in cards:

        judul = card.find("span", class_="snippet__content__title")
        judul = judul.get_text(strip=True) if judul else None

        harga = card.find("div", class_="snippet__content__price")
        harga = harga.get_text(strip=True) if harga else None

        lokasi = card.find("div", class_="snippet__content__location")
        lokasi = lokasi.get_text(" ", strip=True) if lokasi else None

        link = card.find("a")["href"] if card.find("a") else None

        if link:
            link = "https://www.lamudi.co.id" + link

        semua_data.append({
            "Judul": judul,
            "Harga": harga,
            "Lokasi": lokasi,
            "Link": link
        })

driver.quit()

Halaman 61
Card: 30
Halaman 62
Card: 30
Halaman 63
Card: 30
Halaman 64
Card: 30
Halaman 65
Card: 30
Halaman 66
Card: 30
Halaman 67
Card: 30
Halaman 68
Card: 30
Halaman 69
Card: 30
Halaman 70
Card: 30


In [94]:
df = pd.DataFrame(semua_data)

df.head()

,Judul,Harga,Lokasi,Link
0,Apartemen Dijual di Tenggilis Mejoyo,Rp 450Jt,"RW 03, Tenggilis Mejoyo, Tenggilis Mejoyo, Sur...",https://www.lamudi.co.id/properti/41032-73-f58...
1,Apartemen Dijual di Petemon,Rp 220Jt,"RW 01, Petemon, Sawahan, Surabaya, Jawa Timur",https://www.lamudi.co.id/properti/41032-73-196...
2,Apartemen Dijual di Kebayoran Lama,"Rp 7,75M","RW 03, Pondok Pinang, Kebayoran Lama, Jakarta ...",https://www.lamudi.co.id/properti/41032-73-d12...
3,Apartemen Dijual di Ngagel,"Rp 1,50M","RW 02, Ngagel, Wonokromo, Surabaya, Jawa Timur",https://www.lamudi.co.id/properti/41032-73-b36...
4,Apartemen Dijual di Sleman,"Rp 1,20M","area pertokoan, Sariharjo, Ngaglik, Sleman, Da...",https://www.lamudi.co.id/jual/yogyakarta/slema...


In [95]:
df = pd.DataFrame(semua_data)

print(df.shape)

df.to_csv(
    "data/lamudi_apartemen_links_061_070.csv",
    index=False,
    encoding="utf-8-sig"
)

(300, 4)


In [97]:
print("Jumlah baris duplikat:", df.duplicated(subset="Link").sum())

Jumlah baris duplikat: 0


In [98]:
import glob
import pandas as pd

files = glob.glob("data/lamudi_apartemen_links_*.csv")

df = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True
)

df = df.drop_duplicates(subset="Link")

print(df.shape)

df.to_csv(
    "data/lamudi_apartemen_links.csv",
    index=False,
    encoding="utf-8-sig"
)

(2100, 4)


In [99]:
print(df.shape)

(2100, 4)


## **SCRAPING KEDUA (SCRAPING DETAIL)**

In [11]:
import os
import time
import random

import pandas as pd

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [12]:
driver = webdriver.Chrome()

In [13]:
html = driver.page_source

soup = BeautifulSoup(html,"html.parser")

In [14]:
def ambil_text(soup, tag, class_name=None, data_test=None):
    attrs = {}

    if data_test:
        attrs["data-test"] = data_test

    element = soup.find(
        tag,
        class_=class_name,
        attrs=attrs
    )

    return element.get_text(strip=True) if element else None

In [15]:
def ambil_text_multi(soup, selectors):
    for selector in selectors:
        hasil = ambil_text(
            soup,
            tag=selector["tag"],
            class_name=selector.get("class_name"),
            data_test=selector.get("data_test")
        )

        if hasil:
            return hasil

    return None

In [16]:
def scrape_detail(driver, url):

    driver.get(url)

    WebDriverWait(driver,30).until(
        EC.presence_of_element_located((By.TAG_NAME,"h1"))
    )

    soup = BeautifulSoup(driver.page_source,"html.parser")

    # Judul
    judul = ambil_text(
        soup,
        tag="h1"
    )

    # Harga
    harga = ambil_text(
        soup,
        tag="div",
        class_name="prices-and-fees__price",
        data_test="listing-price"
    )

    # Lokasi
    lokasi = soup.find("div", class_="view-map__text")
    lokasi = lokasi.get_text(strip=True) if lokasi else None

    # Kota & Provinsi
    bagian = [x.strip() for x in lokasi.split(",")] if lokasi else []

    kota = bagian[-2] if len(bagian) >= 2 else None
    provinsi = bagian[-1] if len(bagian) >= 1 else None

    # Tipe Properti
    tipe = ambil_text(
        soup,
        tag="span",
        class_name="features-component__value",
        data_test="property-type-value"
    )

    # Kamar Tidur
    kamar_tidur = ambil_text(
        soup,
        tag="div",
        class_name="details-item-value",
        data_test="bedrooms-value"
    )

    # Kamar Mandi
    kamar_mandi = ambil_text(
        soup,
        tag="div",
        class_name="details-item-value",
        data_test="full-bathrooms-value"
    )

    # Luas Total
    luas_total = ambil_text_multi(
        soup,
        [
            {
                "tag": "span",
                "class_name": "features-component__value",
                "data_test": "floor-area-value"
            },
            {
                "tag": "div",
                "class_name": "details-item-value",
                "data_test": "area-value"
            }
        ]
    )

    lantai = ambil_text(
        soup,
        tag="span",
        class_name="features-component__value",
        data_test="floor-value"
    )

    # Tahun Dibangun
    tahun_dibangun = ambil_text(
        soup,
        tag="span",
        class_name="features-component__value",
        data_test="construction-year-value"
    )

    # Sertifikat
    sertifikat = ambil_text(
        soup,
        tag="span",
        class_name="place-features__values",
        data_test="ownership-certificate-value"
    )   

    # Nama Agen
    nama_agen = ambil_text(
        soup,
        tag="span",
        class_name="agency__name",
        data_test="agency-name"
    )

    # Tanggal Posting
    tanggal = soup.find("div", class_="date")
    tanggal = tanggal.get_text(" ", strip=True) if tanggal else None

    if tanggal:
        tanggal = tanggal.split("-")[0].strip()

    return {
        "Judul": judul,
        "Harga": harga,
        "Lokasi": lokasi,
        "Kota": kota,
        "Provinsi": provinsi,
        "Tipe Properti": tipe,
        "Kamar Tidur": kamar_tidur,
        "Kamar Mandi": kamar_mandi,
        "Luas Total": luas_total,
        "Lantai": lantai,
        "Tahun Dibangun": tahun_dibangun,
        "Sertifikat": sertifikat,
        "Nama Agen": nama_agen,
        "Tanggal Posting": tanggal,
        "Link": url
    }

In [17]:
driver = webdriver.Chrome()

data = scrape_detail(
    driver,
    "https://www.lamudi.co.id/properti/41032-73-95cd83cc7304-dbac-19e4a5d-8b75-73a2"
)

driver.quit()

data

{'Judul': 'DI JUAL APARTEMEN 2 BEDROOM SKY HOUSE BSD – READY UNIT HUNI l AKSES AEON MALL',
 'Harga': 'Rp 860Jt',
 'Lokasi': 'BSD City, Pagedangan, Pagedangan, Kabupaten Tangerang, Banten',
 'Kota': 'Kabupaten Tangerang',
 'Provinsi': 'Banten',
 'Tipe Properti': 'Apartemen',
 'Kamar Tidur': '2 Kamar Tidur',
 'Kamar Mandi': '1 kamar mandi',
 'Luas Total': '51 m²',
 'Lantai': '19',
 'Tahun Dibangun': '2023',
 'Sertifikat': 'SHM',
 'Nama Agen': 'Frans Property',
 'Tanggal Posting': '21 Mei 2026',
 'Link': 'https://www.lamudi.co.id/properti/41032-73-95cd83cc7304-dbac-19e4a5d-8b75-73a2'}

In [32]:
import pandas as pd

df = pd.read_csv("data/lamudi_apartemen_links.csv")
print(df.shape)
df.head()

(2100, 4)


,Judul,Harga,Lokasi,Link
0,Apartemen Dijual di Pancoran,Rp 600Jt,"RW 07, Rawajati, Pancoran, Jakarta Selatan, Da...",https://www.lamudi.co.id/properti/41032-73-54c...
1,Apartemen Dijual di Bumi Serpong Damai,Rp 860Jt,"BSD City, Pagedangan, Pagedangan, Kabupaten Ta...",https://www.lamudi.co.id/properti/41032-73-95c...
2,Apartemen Dijual di Bumi Serpong Damai,Rp 850Jt,"BSD City, Pagedangan, Pagedangan, Kabupaten Ta...",https://www.lamudi.co.id/properti/41032-73-6b3...
3,Apartemen Dijual di Bumi Serpong Damai,Rp 840Jt,"BSD Grand CBD, BSD City, Pagedangan, Pagedanga...",https://www.lamudi.co.id/properti/41032-73-246...
4,Apartemen Dijual di Balikpapan Selatan,Rp 600Jt,"Sepinggan Baru, Balikpapan Selatan, Balikpapan...",https://www.lamudi.co.id/properti/41032-73-f74...


In [41]:
TOTAL_DATA = 2100

driver = webdriver.Chrome()

driver.set_page_load_timeout(30)

hasil = []

start_time = time.time()

In [42]:
AUTOSAVE = 100

In [43]:
for i, url in enumerate(df["Link"].head(TOTAL_DATA), start=1):

    try:

        print(f"[{i}/{TOTAL_DATA}]")

        data = scrape_detail(driver, url)

        hasil.append(data)

        #Autosave setiap 100 data
        if i % AUTOSAVE == 0:

            hasil_df = pd.DataFrame(hasil)

            hasil_df.to_csv(
                "data/lamudi_apartemen_detail_temp.csv",
                index=False,
                encoding="utf-8-sig"
            )

            elapsed = time.time() - start_time
            avg = elapsed / i
            remaining = (TOTAL_DATA - i) * avg

            print("=" * 60)
            print(f"Autosave {i} data")
            print(f"Waktu berjalan : {elapsed/60:.2f} menit")
            print(f"Rata-rata : {avg:.2f} detik/data")
            print(f"Estimasi sisa : {remaining/60:.2f} menit")
            print("=" * 60)

        time.sleep(random.uniform(1,2))

    except Exception as e:

        print("=" * 60)
        print(f"Gagal data ke-{i}")
        print(url)
        print(e)
        print("=" * 60)

driver.quit()

print("Browser ditutup")

[1/2100]
[2/2100]
[3/2100]
[4/2100]
[5/2100]
[6/2100]
[7/2100]
[8/2100]
[9/2100]
[10/2100]
[11/2100]
[12/2100]
[13/2100]
[14/2100]
[15/2100]
[16/2100]
[17/2100]
[18/2100]
[19/2100]
[20/2100]
[21/2100]
[22/2100]
[23/2100]
[24/2100]
[25/2100]
[26/2100]
[27/2100]
[28/2100]
[29/2100]
[30/2100]
[31/2100]
[32/2100]
[33/2100]
[34/2100]
[35/2100]
[36/2100]
[37/2100]
[38/2100]
[39/2100]
[40/2100]
[41/2100]
[42/2100]
[43/2100]
[44/2100]
[45/2100]
[46/2100]
[47/2100]
[48/2100]
[49/2100]
[50/2100]
[51/2100]
[52/2100]
[53/2100]
[54/2100]
[55/2100]
[56/2100]
[57/2100]
[58/2100]
[59/2100]
[60/2100]
[61/2100]
[62/2100]
[63/2100]
[64/2100]
[65/2100]
[66/2100]
[67/2100]
[68/2100]
[69/2100]
[70/2100]
[71/2100]
[72/2100]
[73/2100]
[74/2100]
[75/2100]
[76/2100]
[77/2100]
[78/2100]
[79/2100]
[80/2100]
[81/2100]
[82/2100]
[83/2100]
[84/2100]
[85/2100]
[86/2100]
[87/2100]
[88/2100]
[89/2100]
[90/2100]
[91/2100]
[92/2100]
[93/2100]
[94/2100]
[95/2100]
[96/2100]
[97/2100]
[98/2100]
[99/2100]
[100/2100]
Autosave

In [47]:
hasil_df = pd.DataFrame(hasil)

print(hasil_df.shape)

hasil_df.head()

(2095, 15)


,Judul,Harga,Lokasi,Kota,Provinsi,Tipe Properti,Kamar Tidur,Kamar Mandi,Luas Total,Lantai,Tahun Dibangun,Sertifikat,Nama Agen,Tanggal Posting,Link
0,DIJUAL CEPAT UNIT STUDIO TOWER MATOA LT. 17 WO...,Rp 600Jt,"RW 07, Rawajati, Pancoran, Jakarta Selatan, Da...",Jakarta Selatan,Daerah Khusus Ibukota Jakarta,Apartemen,1 kamar tidur,1 kamar mandi,28 m²,17,2015,PPJB,MoFu,4 Jun 2026,https://www.lamudi.co.id/properti/41032-73-54c...
1,DI JUAL APARTEMEN 2 BEDROOM SKY HOUSE BSD – RE...,Rp 860Jt,"BSD City, Pagedangan, Pagedangan, Kabupaten Ta...",Kabupaten Tangerang,Banten,Apartemen,2 Kamar Tidur,1 kamar mandi,51 m²,19,2023,SHM,Frans Property,21 Mei 2026,https://www.lamudi.co.id/properti/41032-73-95c...
2,"Di jual 2BR Sky House BSD city, lokasi premium...",Rp 850Jt,"BSD City, Pagedangan, Pagedangan, Kabupaten Ta...",Kabupaten Tangerang,Banten,Apartemen,2 Kamar Tidur,1 kamar mandi,51 m²,20,2023,SHM,Frans Property,2 Jun 2026,https://www.lamudi.co.id/properti/41032-73-6b3...
3,INVESTASI TERBAIK 2026: 2BR SAMPING AEON BSD H...,Rp 840Jt,"BSD Grand CBD, BSD City, Pagedangan, Pagedanga...",Kabupaten Tangerang,Banten,Apartemen,2 Kamar Tidur,1 kamar mandi,48 m²,26,2027,SHM,Sky House BSD.,3 Jun 2026,https://www.lamudi.co.id/properti/41032-73-246...
4,Dijual Apartemen Sepinggan Terrace Balikpapan,Rp 600Jt,"Sepinggan Baru, Balikpapan Selatan, Balikpapan...",Balikpapan,Kalimantan Timur,Apartemen,NaN,1 kamar mandi,29 m²,NaN,NaN,NaN,Frans Property,10 Nov 2025,https://www.lamudi.co.id/properti/41032-73-f74...


In [48]:
hasil_df.to_csv(
    "data/lamudi_apartemen_detail.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Berhasil disimpan")

Berhasil disimpan


In [49]:
hasil_df.info()

hasil_df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 2095 entries, 0 to 2094
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Judul            2090 non-null   str  
 1   Harga            2090 non-null   str  
 2   Lokasi           2090 non-null   str  
 3   Kota             2087 non-null   str  
 4   Provinsi         2087 non-null   str  
 5   Tipe Properti    2090 non-null   str  
 6   Kamar Tidur      2033 non-null   str  
 7   Kamar Mandi      2039 non-null   str  
 8   Luas Total       2043 non-null   str  
 9   Lantai           1576 non-null   str  
 10  Tahun Dibangun   426 non-null    str  
 11  Sertifikat       913 non-null    str  
 12  Nama Agen        2090 non-null   str  
 13  Tanggal Posting  2090 non-null   str  
 14  Link             2095 non-null   str  
dtypes: str(15)
memory usage: 245.6 KB


Judul                 5
Harga                 5
Lokasi                5
Kota                  8
Provinsi              8
Tipe Properti         5
Kamar Tidur          62
Kamar Mandi          56
Luas Total           52
Lantai              519
Tahun Dibangun     1669
Sertifikat         1182
Nama Agen             5
Tanggal Posting       5
Link                  0
dtype: int64